# One-Lane Traffic (Mesa 3.3)

This notebook targets **Mesa 3.3**. A row of *car* agents moves along a circular one-lane road. The
example uses the stable `discrete_space` grid, bulk agent creation with `create_agents`, a
`DataCollector` for model-level measurements, `batch_run` with `iterations=`, and the visualization
API introduced in Mesa 3.3 (`SpaceRenderer` together with `AgentPortrayalStyle`).

In [ ]:
# Environment pin (uncomment to install):
#!pip install mesa[rec]==3.3.0

In [ ]:
import mesa
from mesa.discrete_space import CellAgent, OrthogonalVonNeumannGrid

In [ ]:
class Car(CellAgent):
    """A car that advances to one of its road neighbours each step."""

    def __init__(self, model, cell):
        super().__init__(model)
        self.cell = cell

    def step(self):
        self.cell = self.cell.neighborhood.select_random_cell()

In [ ]:
class TrafficModel(mesa.Model):
    def __init__(self, n=10, length=20, seed=None):
        super().__init__(seed=seed)
        self.grid = OrthogonalVonNeumannGrid((length, 1), torus=True, random=self.random)
        Car.create_agents(
            self, n,
            cell=self.random.choices(self.grid.all_cells.cells, k=n),
        )
        self.datacollector = mesa.DataCollector(
            model_reporters={"NumCars": lambda m: len(m.agents)}
        )

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)

In [ ]:
model = TrafficModel(seed=7)
for _ in range(20):
    model.step()

model.datacollector.get_model_vars_dataframe().tail()

In [ ]:
results = mesa.batch_run(
    TrafficModel,
    parameters={"n": [5, 10]},
    iterations=3,
    max_steps=20,
)
print(f"Collected {len(results)} runs.")

## Visualization

Mesa 3.3 introduces `SpaceRenderer`. The road is rendered by a renderer whose agent styling is
provided by a function returning an `AgentPortrayalStyle`, and the renderer is passed to `SolaraViz`.

In [ ]:
import warnings

# The Solara stack bundled with Mesa 3.3 emits a hook-validation UserWarning the first time the
# visualization package is imported. It concerns Mesa's own component code, not this notebook, so we
# silence it locally to keep the import clean.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    from mesa.visualization import SolaraViz, SpaceRenderer, make_plot_component
    from mesa.visualization.components import AgentPortrayalStyle


def agent_portrayal(agent):
    return AgentPortrayalStyle(color="red", marker="o", size=20)


renderer = SpaceRenderer(TrafficModel(), backend="matplotlib").render(agent_portrayal)
page = SolaraViz(TrafficModel(), renderer, components=[], name="Traffic")